# 03 · Silver

Bronze → cleaned, typed and deduplicated **Delta table** at `data/silver/enem_cleaned`.

1. Drop columns not used downstream (exam answers, answer keys, socioeconomic questionnaire).
2. Rename to English snake_case.
3. Cast types: scores → `double` (INEP uses `,` or `.` as decimal separator), codes → `int`.
4. Keep only the latest record per `registration_id`.
5. Write with a Delta **`MERGE`** (upsert): re-running never duplicates candidates.

In [4]:
%run ./00_setup.ipynb

:: loading settings :: url = jar:file:/home/gabriel/Documents/GitHub/ETL-com-PySpark/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/gabriel/Documents/GitHub/ETL-com-PySpark/.ivy2/cache
The jars for the packages stored in: /home/gabriel/Documents/GitHub/ETL-com-PySpark/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-707698f9-ff6b-466c-a877-36a4e07566e6;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 91ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-------------------------------------

Spark 3.5.3 ready


In [5]:
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql.functions import col, regexp_replace, row_number, trim

DROP_COLUMNS = (
    [f"TX_RESPOSTAS_{area}" for area in ("CN", "CH", "LC", "MT")]    # exam answers
    + [f"TX_GABARITO_{area}" for area in ("CN", "CH", "LC", "MT")]   # answer keys
    + [f"Q{i:03d}" for i in range(1, 26)]                            # questionnaire Q001-Q025
)

RENAME_COLUMNS = {
    "NU_INSCRICAO": "registration_id",
    "NU_ANO": "year",
    "TP_FAIXA_ETARIA": "age_group",
    "TP_SEXO": "gender",
    "TP_COR_RACA": "race",
    "TP_ST_CONCLUSAO": "high_school_status",
    "TP_ANO_CONCLUIU": "high_school_completion_year",
    "TP_ESCOLA": "school_type",
    "TP_ENSINO": "education_type",
    "IN_TREINEIRO": "is_trainee",
    "SG_UF_PROVA": "exam_state",
    "TP_PRESENCA_CN": "presence_natural_sciences",
    "TP_PRESENCA_CH": "presence_humanities",
    "TP_PRESENCA_LC": "presence_languages",
    "TP_PRESENCA_MT": "presence_math",
    "NU_NOTA_CN": "score_natural_sciences",
    "NU_NOTA_CH": "score_humanities",
    "NU_NOTA_LC": "score_languages",
    "NU_NOTA_MT": "score_math",
    "TP_LINGUA": "foreign_language",
    "TP_STATUS_REDACAO": "essay_status",
    "NU_NOTA_COMP1": "essay_comp1",
    "NU_NOTA_COMP2": "essay_comp2",
    "NU_NOTA_COMP3": "essay_comp3",
    "NU_NOTA_COMP4": "essay_comp4",
    "NU_NOTA_COMP5": "essay_comp5",
    "NU_NOTA_REDACAO": "score_essay",
}

# Types derived from the new names: text ids, decimal scores, everything else is an integer code
STRING_COLUMNS = {"registration_id", "gender", "exam_state"}
DOUBLE_COLUMNS = {c for c in RENAME_COLUMNS.values() if c.startswith(("score_", "essay_comp"))}
INT_COLUMNS = set(RENAME_COLUMNS.values()) - STRING_COLUMNS - DOUBLE_COLUMNS

In [6]:
df = (
    spark.read.format("delta").load(BRONZE_TABLE)
    .drop(*DROP_COLUMNS)
    .withColumnsRenamed(RENAME_COLUMNS)
)

df = df.withColumns(
    {c: trim(col(c)) for c in STRING_COLUMNS}
    | {c: col(c).cast("int") for c in INT_COLUMNS}
    | {c: regexp_replace(col(c), ",", ".").cast("double") for c in DOUBLE_COLUMNS}
)

latest_first = Window.partitionBy("registration_id").orderBy(col("_ingestion_timestamp").desc())
df_silver = (
    df.filter(col("registration_id").isNotNull() & (col("registration_id") != ""))
    .withColumn("_row", row_number().over(latest_first))
    .filter(col("_row") == 1)
    .drop("_row")
)

Data quality checks before writing: the cell fails if a rule is broken.

In [7]:
total = df_silver.count()
assert total > 0, "Silver is empty"
assert df_silver.select("registration_id").distinct().count() == total, "Duplicated registration_id"
assert df_silver.filter((col("score_math") < 0) | (col("score_math") > 1000)).isEmpty(), "Math score out of range"
print(f"Quality checks passed: {total} candidates")

26/11/25 14:21:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Quality checks passed: 20 candidates


In [8]:
if DeltaTable.isDeltaTable(spark, SILVER_TABLE):
    (
        DeltaTable.forPath(spark, SILVER_TABLE).alias("t")
        .merge(df_silver.alias("s"), "t.registration_id = s.registration_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_silver.write.format("delta").partitionBy("year", "exam_state").save(SILVER_TABLE)

In [9]:
silver = spark.read.format("delta").load(SILVER_TABLE)
print(f"Silver: {silver.count()} rows | {len(silver.columns)} columns")
silver.select("registration_id", "year", "exam_state", "gender", "score_math", "score_essay").show(5)

Silver: 20 rows | 29 columns
+---------------+----+----------+------+----------+-----------+
|registration_id|year|exam_state|gender|score_math|score_essay|
+---------------+----+----------+------+----------+-----------+
|   200001000009|2020|        CE|     F|     710.8|      780.0|
|   200001000018|2020|        CE|     M|     690.4|      760.0|
|   200001000003|2020|        MG|     F|     740.5|      860.0|
|   200001000013|2020|        MG|     F|     510.4|      560.0|
|   200001000002|2020|        RJ|     M|     530.7|      620.0|
+---------------+----+----------+------+----------+-----------+
only showing top 5 rows



### Delta Lake: history and time travel
Run this notebook twice: the second run shows up as a `MERGE` version in the history.

In [10]:
spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_TABLE}`").select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

+-------+-----------------------+---------+--------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                              |
+-------+-----------------------+---------+--------------------------------------------------------------+
|0      |2026-11-25 14:22:01.389|WRITE    |{numFiles -> 10, numOutputRows -> 20, numOutputBytes -> 84650}|
+-------+-----------------------+---------+--------------------------------------------------------------+



In [11]:
# Read the table exactly as it was in its first version
silver_v0 = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_TABLE)
print(f"Version 0: {silver_v0.count()} rows | current: {silver.count()} rows")

Version 0: 20 rows | current: 20 rows


In [12]:
spark.stop()